# Feature Extraction — Brain Invaders (bi2015a)

Loads preprocessed epochs from `01_preprocessing.ipynb` and extracts feature sets for P300 classification.

**Research question:** Do delta (0.5–3 Hz) and theta (3–7 Hz) ERP subcomponents differ in cross-subject P300 classification performance, and does that difference depend on flash duration (50 / 80 / 110 ms)?

| Feature key | Description | Channels | Shape |
|-------------|-------------|----------|-------|
| **`erp`** | Raw epoch waveform, post-stimulus (≥0 s), all 32 ch, ds×16 | 32 | `(n, 1056)` |
| **`delta_wave`** | Bandpass 0.5–3 Hz (butter order 4, filtfilt), ds×16 | 9 (Cz, CP1/2/5/6, Pz, P3/4/8) | `(n, 297)` |
| **`theta_wave`** | Bandpass 3–7 Hz (butter order 4, filtfilt), ds×16 | 6 (AFz, FC1/2, F3/4, Cz) | `(n, 198)` |
| **`dt_wave`** | delta_wave + theta_wave concatenated | — | `(n, 495)` |

Bandpass filtering uses a 4th-order zero-phase Butterworth filter (`scipy.signal.butter` + `filtfilt`) applied to the full epoch (all channels, all timepoints). The filtered signal is then channel-selected, cropped to post-stimulus (≥0 s), downsampled ×16, and flattened.

## 1. Imports

| Library | Alias | Purpose in this notebook |
|---------|-------|--------------------------|
| `numpy` | `np` | Numerical arrays — all EEG data manipulation, downsampling, and feature vectors |
| `pandas` | `pd` | Tabular data — reading epoch metadata (target labels) |
| `matplotlib.pyplot` | `plt` | All plotting |
| `matplotlib.gridspec.GridSpec` | — | Advanced subplot layout with custom grid sizes |
| `mne` | — | Loads `.fif` epoch files and provides channel-selection utilities |
| `scipy.signal.butter` | — | Designs Butterworth bandpass filter coefficients |
| `scipy.signal.filtfilt` | — | Applies the filter in zero-phase mode (forward + backward pass) |
| `pathlib.Path` | — | Cross-platform file path construction |

`%matplotlib inline` renders figures inside the notebook.  
`mne.set_log_level('WARNING')` silences MNE's verbose progress messages.

In [ ]:
%matplotlib inline                          # render figures inside the notebook
import numpy as np                          # numerical arrays — all EEG data manipulation
import pandas as pd                         # tabular data — reading epoch metadata (labels)
import matplotlib.pyplot as plt             # all plotting
from matplotlib.gridspec import GridSpec    # advanced subplot layout
import mne                                  # loads .fif epoch files; channel-selection utilities
from scipy.signal import butter, filtfilt   # Butterworth filter design and zero-phase application
from pathlib import Path                    # cross-platform file path construction

mne.set_log_level('WARNING')                # silence MNE's verbose progress messages

## 2. Configuration

**Change `SUBJECT` to process a different participant.** All file paths and channel indices are derived automatically.

### Channel subsets for bandpass features

Two frequency bands capture different aspects of the P300 response, each strongest at a different part of the scalp — so each band uses a different electrode subset:

**Delta channels (0.5–3 Hz) — posterior / centro-parietal:**  
`Cz, CP1, CP2, CP5, CP6, Pz, P3, P4, P8`  
The P300 is a slow positive wave that is maximal at parietal and central-parietal electrodes. Filtering to 0.5–3 Hz isolates this slow component cleanly from faster oscillations. Using posterior channels captures where the signal is strongest and avoids adding noisy frontal channels where delta activity carries little P300 information.

**Theta channels (3–7 Hz) — frontal / fronto-central:**  
`AFz, FC1, FC2, F3, F4, Cz`  
When the brain detects an attended target, frontal regions (prefrontal cortex, anterior cingulate) produce a burst of theta-band activity linked to working-memory updating and attentional gating. This is *complementary* to the posterior delta wave — together they represent both the "sensory recognition" and "cognitive significance" signals of the P300 response.

### Bandpass filter parameters

| Parameter | Value | Why |
|-----------|-------|-----|
| `SFREQ` | 512 Hz | Amplifier sampling rate — must match the preprocessing step |
| `DELTA_LOW / HIGH` | 0.5–3 Hz | Delta band: captures the slow P300 wave without DC drift (hence 0.5 not 0) |
| `THETA_LOW / HIGH` | 3–7 Hz | Theta band: captures frontal attentional theta without overlap with delta |
| `BP_ORDER` | 4 | Butterworth order: steeper roll-off than order 2, no instability unlike order 8+ |

The file-check loop at the bottom of the config cell confirms that all three `.fif` files from `01_preprocessing.ipynb` exist before the main loop tries to load them.

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
SUBJECT     = 2   # participant number (1–43); avoid 1 and 27 (bad recordings)
_PROJECT_ROOT = Path().resolve()
DATA_ROOT   = _PROJECT_ROOT / "data" / "raw" / f"subject_{SUBJECT:02d}_csv"
# :02d zero-pads single-digit numbers (e.g. 2 → "02"), matching the filenames on disk
OUTPUT_ROOT = _PROJECT_ROOT / "data" / "preprocessed"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)   # create folder if it doesn't exist yet

# Flash duration per session (ms) — from Table 2 of the bi2015a paper
FLASH_DURATION_MS = {1: 110, 2: 80, 3: 50}   # session 1 slowest, session 3 fastest

SESSIONS = [1, 2, 3]

# Delta channels: posterior/centro-parietal — where the P300 slow wave is strongest
DELTA_CHANNELS = ['Cz', 'CP1', 'CP2', 'CP5', 'CP6', 'Pz', 'P3', 'P4', 'P8']   # 9 channels
# Theta channels: frontal/fronto-central — where attentional theta bursts are strongest
THETA_CHANNELS = ['AFz', 'FC1', 'FC2', 'F3', 'F4', 'Cz']                       # 6 channels

# ── Bandpass filter parameters ────────────────────────────────────────────────
SFREQ      = 512.0   # amplifier sampling rate (must match preprocessing)
DELTA_LOW  = 0.5     # Hz — lower edge of delta band (0.5 avoids DC drift artefacts)
DELTA_HIGH = 3.0     # Hz — upper edge of delta band
THETA_LOW  = 3.0     # Hz — lower edge of theta band
THETA_HIGH = 7.0     # Hz — upper edge of theta band
BP_ORDER   = 4       # Butterworth filter order: steeper roll-off than 2, stable unlike 8+

# ── VERIFY FILES EXIST ────────────────────────────────────────────────────────
print(f"Data root      : {DATA_ROOT}")
print(f"Output root    : {OUTPUT_ROOT}")
print(f"Subject        : {SUBJECT}")
print(f"Delta channels : {DELTA_CHANNELS}  band={DELTA_LOW}–{DELTA_HIGH} Hz  order={BP_ORDER}")
print(f"Theta channels : {THETA_CHANNELS}  band={THETA_LOW}–{THETA_HIGH} Hz  order={BP_ORDER}")
print()
for s in SESSIONS:
    fif = OUTPUT_ROOT / f"subject_{SUBJECT:02d}_session_{s:02d}_epo.fif"
    status = 'OK' if fif.exists() else 'MISSING — run 01_preprocessing.ipynb first'
    print(f"  Session {s} ({FLASH_DURATION_MS[s]:3d} ms): [{status}]  {fif.name}")

## 3. Helper Functions

Two reusable functions that handle the core feature engineering. Both operate on NumPy arrays of shape `(n_epochs, n_channels, n_times)`.

---

### `extract_erp(epoch_data, times, downsample_factor, channel_indices)`

Converts a 3-D epoch array into a 2-D feature matrix that a scikit-learn classifier can consume. Three steps:

1. **Crop to post-stimulus (t ≥ 0):** The 200 ms pre-stimulus window was used for baseline correction during preprocessing and contains no ERP signal. Removing it prevents the classifier from fitting to baseline noise and reduces the feature dimension roughly by 1/6 (200 ms vs 1000 ms of post-stimulus).

2. **Channel selection (optional):** For the broadband ERP, all 32 channels are kept. For delta and theta features, only the anatomically appropriate subset is kept (see Configuration). Fewer channels → shorter feature vector → faster classifier training and less risk of overfitting.

3. **Downsample × 16, then flatten:** At 512 Hz, keeping every 16th sample gives an effective rate of 32 Hz — still well above the Nyquist frequency for theta (7 Hz requires > 14 Hz), so no signal information is lost. `reshape(n_epochs, -1)` flattens each 2-D `(channels × downsampled_time)` matrix into a single 1-D row vector per epoch. Channel order is preserved: all time points for channel 0, then all for channel 1, and so on.

---

### `bandpass_filter(data, sfreq, low, high, order)`

Isolates one frequency band from the broadband EEG. Two key design choices:

- **Butterworth filter** — its passband is maximally flat (no amplitude ripple), so ERP component amplitudes within the band are not distorted.
- **`filtfilt` (zero-phase filtering)** — the filter is applied *twice*: once forward in time, once backward. The phase shifts cancel exactly, so P300 peak latencies (~300 ms) remain at their true timing. A single-pass filter would shift the signal by tens of milliseconds — fatal when the timing of the P300 is what the classifier relies on.

`butter()` works in normalised frequency (0–1, where 1 = Nyquist = sfreq/2). Dividing by `sfreq/2` converts the Hz cutoffs into this normalised scale.

In [ ]:
def extract_erp(epoch_data, times, downsample_factor=16, channel_indices=None):
    """Crop to post-stimulus window (≥ 0 s), optionally restrict channels, downsample, flatten.
    epoch_data : (n_epochs, n_channels, n_times) → (n_epochs, n_ch * n_times_ds)
    """
    # Step 1: keep only post-stimulus samples.
    # times is a 1-D array; times >= 0 produces a boolean mask that is True at every
    # sample from stimulus onset onward. The pre-stimulus period contains no ERP signal.
    mask    = times >= 0
    cropped = epoch_data[:, :, mask]   # shape: (n_epochs, n_channels, n_post_times)

    # Step 2: restrict to a channel subset if requested.
    # channel_indices must be integer positions (not channel names) for numpy slicing.
    # channel_indices=None keeps all channels (used for the broadband ERP feature).
    if channel_indices is not None:
        cropped = cropped[:, channel_indices, :]   # shape: (n_epochs, n_selected_ch, n_post_times)

    # Step 3: downsample by keeping every Nth sample, then flatten to a 1-D vector per epoch.
    # ::downsample_factor means "take every 16th sample" — at 512 Hz that gives 32 Hz.
    # .reshape(n_epochs, -1) collapses (channels, time) into a single row per epoch.
    # The -1 lets NumPy compute the total number of features automatically.
    return cropped[:, :, ::downsample_factor].reshape(len(epoch_data), -1)


def bandpass_filter(data, sfreq, low, high, order=4):
    """Zero-phase Butterworth bandpass filter applied along the time axis.
    data : (n_epochs, n_channels, n_times)
    Returns filtered array of the same shape.
    """
    # butter() works in normalised frequency (0.0–1.0), where 1.0 = Nyquist = sfreq/2.
    # Dividing by nyq converts the Hz cutoffs into this normalised scale.
    nyq  = sfreq / 2.0
    # butter() returns b (numerator) and a (denominator) filter coefficients.
    b, a = butter(order, [low / nyq, high / nyq], btype='band')

    # filtfilt applies the filter forward then backward in time.
    # The two phase shifts cancel exactly → zero net delay → P300 peak timing is preserved.
    # axis=2 filters along the time axis; each (epoch, channel) pair is processed independently.
    return filtfilt(b, a, data, axis=2)

## 4. Main Loop — Load, Extract, Save

Iterates over all three sessions for the selected subject. For each session the loop:

1. **Checks** the `.fif` file exists and contains at least 20 epochs — too few epochs makes classifier training unreliable
2. **Loads** the MNE epoch file produced by `01_preprocessing.ipynb`
3. **Builds integer channel-index lists** — NumPy array slicing requires integer positions, not channel name strings, so we pre-compute the index of each channel name once
4. **Computes four feature representations** from the same epoch data (see table below)
5. **Prints a shape summary** so you can confirm the dimensions are as expected
6. **Saves** all feature sets to a single `.npz` file — one file per session, self-contained
7. **Stores** results in `all_sessions` so the visualisation cells that follow can redraw plots without re-running the pipeline

The `all_sessions` dictionary is keyed by session number (1, 2, 3). Each entry holds the raw data array, the delta- and theta-filtered arrays, the label vector, and the channel-index lists needed to re-create every plot.

### Feature sets saved per session

| Key | Description | Shape |
|-----|-------------|-------|
| `erp` | Raw waveform, all 32 ch, post-stimulus (≥ 0 s), ds×16 | `(n, 1056)` |
| `delta_wave` | 0.5–3 Hz bandpass, 9 posterior ch, ds×16 | `(n, 297)` |
| `theta_wave` | 3–7 Hz bandpass, 6 frontal ch, ds×16 | `(n, 198)` |
| `dt_wave` | delta_wave + theta_wave concatenated | `(n, 495)` |
| `labels` | Binary target labels (1 = target, 0 = non-target) | `(n,)` |
| `flash_ms` | Flash duration for this session | scalar |

**How the shapes are computed:**  
The post-stimulus window is 1.0 s at 512 Hz = 513 time samples (t = 0 to +1.0 s inclusive). After ×16 downsampling: ⌈513/16⌉ = 33 samples per channel.  
- ERP: 32 ch × 33 = **1056**  
- Delta: 9 ch × 33 = **297**  
- Theta: 6 ch × 33 = **198**  
- DT: 297 + 198 = **495**

In [ ]:
all_sessions = {}   # stores raw data, filtered signals, and metadata per session — used by visualisation cells below

for SESSION in SESSIONS:
    fif_path  = OUTPUT_ROOT / f"subject_{SUBJECT:02d}_session_{SESSION:02d}_epo.fif"
    save_path = OUTPUT_ROOT / f"subject_{SUBJECT:02d}_session_{SESSION:02d}_features.npz"
    flash_ms  = FLASH_DURATION_MS[SESSION]

    # ── 4a. Load ────────────────────────────────────────────────────────────────
    if not fif_path.exists():
        print(f"WARNING: {fif_path.name} not found — skipping session {SESSION}")
        continue

    epochs = mne.read_epochs(fif_path, verbose=False)   # load the .fif produced by 01_preprocessing

    if len(epochs) == 0:
        print(f"  WARNING: Subject {SUBJECT} Session {SESSION} has 0 epochs — skipping")
        continue
    if len(epochs) < 20:
        # Too few epochs for reliable classifier training — skip rather than produce unreliable features
        print(f"  WARNING: Subject {SUBJECT} Session {SESSION} has only {len(epochs)} epochs — skipping")
        continue

    # ── 4b. Extract raw data ──────────────────────────────────────────────────
    # pick_types returns integer indices of channels matching the criteria:
    # eeg=True keeps EEG channels; stim=False excludes the STIM trigger channel
    eeg_picks    = mne.pick_types(epochs.info, eeg=True, stim=False)
    eeg_ch_names = [epochs.ch_names[i] for i in eeg_picks]   # ordered list of EEG channel names
    eeg_info     = mne.pick_info(epochs.info, eeg_picks)      # sub-Info object for these channels (used for topomap plotting)

    # Pre-compute integer indices so that numpy array slicing is fast and unambiguous.
    # .index(ch) finds the position of channel name ch in the ordered list.
    cz_idx        = eeg_ch_names.index('Cz')
    delta_indices = [eeg_ch_names.index(ch) for ch in DELTA_CHANNELS]   # list of 9 ints
    theta_indices = [eeg_ch_names.index(ch) for ch in THETA_CHANNELS]   # list of 6 ints

    data   = epochs.get_data(picks='eeg')       # shape: (n_epochs, 32, n_times)
    times  = epochs.times                       # 1-D array of time values in seconds; t=0 = stimulus onset
    labels = epochs.metadata['target'].values   # 1 = target flash, 0 = non-target flash

    # ── 4c. Extract features ──────────────────────────────────────────────────
    # Feature set 1: broadband ERP — post-stimulus, all 32 channels, ds×16, flattened
    features_erp = extract_erp(data, times)

    # Bandpass-filter the full epoch (including pre-stimulus) into delta and theta bands.
    # Filtering the full window first avoids edge artefacts that would appear if we
    # cropped to post-stimulus before filtering.
    data_delta = bandpass_filter(data, SFREQ, DELTA_LOW, DELTA_HIGH, BP_ORDER)
    data_theta = bandpass_filter(data, SFREQ, THETA_LOW, THETA_HIGH, BP_ORDER)

    # Feature sets 2 & 3: select the anatomically appropriate channel subset, crop, ds×16, flatten
    features_delta_wave = extract_erp(data_delta, times, channel_indices=delta_indices)
    features_theta_wave = extract_erp(data_theta, times, channel_indices=theta_indices)

    # Feature set 4: concatenate delta and theta into one longer vector per epoch.
    # np.hstack stacks arrays column-wise: (n, 297) + (n, 198) → (n, 495)
    features_dt_wave    = np.hstack([features_delta_wave, features_theta_wave])

    # ── 4d. Print summary ─────────────────────────────────────────────────────
    n_target    = int(labels.sum())             # sum works because 1=target, 0=non-target
    n_nontarget = len(labels) - n_target
    print(f"Session {SESSION} ({flash_ms} ms) — {len(epochs)} epochs ({n_target} target, {n_nontarget} non-target)")
    print(f"  ERP shape        : {features_erp.shape}")
    print(f"  Delta wave shape : {features_delta_wave.shape}")
    print(f"  Theta wave shape : {features_theta_wave.shape}")
    print(f"  DT wave shape    : {features_dt_wave.shape}")

    # ── 4e. Save features ─────────────────────────────────────────────────────
    # np.savez writes multiple named arrays into one compressed .npz file.
    # Each keyword argument becomes a named key: np.load(path)['erp'], etc.
    # Storing subject, session, and flash_ms makes each file self-contained.
    np.savez(
        save_path,
        erp        = features_erp,
        delta_wave = features_delta_wave,
        theta_wave = features_theta_wave,
        dt_wave    = features_dt_wave,
        labels     = labels,
        flash_ms   = flash_ms,
        subject    = SUBJECT,
        session    = SESSION,
    )
    print(f"  Saved → {save_path}")
    print()

    # Store raw data and filtered signals for the visualisation cells below.
    # Keeping both the original and filtered arrays avoids re-running the bandpass filters.
    all_sessions[SESSION] = dict(
        flash_ms      = flash_ms,
        data          = data,         # (n_epochs, 32, n_times) — broadband, post-preprocessing
        times         = times,
        labels        = labels,
        eeg_info      = eeg_info,
        cz_idx        = cz_idx,
        delta_indices = delta_indices,
        theta_indices = theta_indices,
        data_delta    = data_delta,   # (n_epochs, 32, n_times) — delta-filtered (0.5–3 Hz)
        data_theta    = data_theta,   # (n_epochs, 32, n_times) — theta-filtered (3–7 Hz)
    )

## 5. Per-Session Visualisation

For each of the three sessions, three time-series panels verify that the feature extraction captured real, discriminative neural signals.

**Why compare target vs non-target in every plot?**  
The entire BCI premise rests on the assumption that target and non-target epochs look *different* in the EEG. If the red and blue curves are indistinguishable, the feature set contains no discriminative signal and any classifier trained on it will perform at chance. These plots are the primary sanity check before moving to classification.

---

**Panel 1 — ERP at Cz (broadband, 0.1–30 Hz)**  
The average event-related potential at electrode Cz (central midline, top of the head). This is the same broadband ERP from `01_preprocessing.ipynb`, now shown as a reference against which the band-limited panels below can be compared.  
- **What to look for:** A clear positive peak in the red (target) curve between ~300–600 ms — the **P300** — that is absent or much weaker in the blue (non-target) curve.  
- **Why Cz?** Cz sits at the intersection of the central and midline axes where the P300 is typically maximal in P300-based BCIs.

**Panel 2 — Delta wave at Pz (0.5–3 Hz bandpass)**  
The same epoch data passed through the delta bandpass filter and averaged at electrode Pz (parietal midline, ~10 cm behind Cz). Delta filtering removes all fast oscillations, leaving only the very slow envelope of the ERP.  
- **What to look for:** A broad, slow positive hump in the target average that peaks *later* than the broadband P300 (around 500–900 ms) and is near zero for non-targets. This is the **late slow wave** — delta-band activity driven by the cognitive significance of the attended stimulus.  
- **Why Pz?** The slow posterior positivity linked to the P300 is spatially broad but strongest at parietal midline sites.

**Panel 3 — Theta wave at AFz (3–7 Hz bandpass)**  
Epoch data after the theta bandpass filter, averaged at AFz (anterior frontal midline, just above the forehead).  
- **What to look for:** An *oscillatory* burst in the target curve peaking around 300–500 ms. Frontal theta reflects **attentional gating and working-memory updating** — the prefrontal process of recognising the target and encoding it. Non-target epochs should show little theta modulation at this site.  
- **Why AFz?** Frontal theta is generated by prefrontal and anterior cingulate cortex and is maximal over frontal midline electrodes (AFz, Fz).

| Panel | Signal | Electrode | Expected target effect | Typical timing |
|-------|--------|-----------|------------------------|----------------|
| ERP | Broadband (0.1–30 Hz) | Cz | Positive P300 peak | 300–600 ms |
| Delta wave | 0.5–3 Hz | Pz | Slow positive hump | 500–900 ms |
| Theta wave | 3–7 Hz | AFz | Oscillatory burst | 300–500 ms |

In [ ]:
# Positions of the representative electrodes within their channel-subset lists
pz_in_delta  = DELTA_CHANNELS.index('Pz')    # which slot in DELTA_CHANNELS holds 'Pz'
afz_in_theta = THETA_CHANNELS.index('AFz')   # which slot in THETA_CHANNELS holds 'AFz'

for SESSION in sorted(all_sessions):
    sess          = all_sessions[SESSION]
    flash_ms      = sess['flash_ms']
    data          = sess['data']
    times         = sess['times']
    labels        = sess['labels']
    cz_idx        = sess['cz_idx']
    delta_indices = sess['delta_indices']
    theta_indices = sess['theta_indices']
    data_delta    = sess['data_delta']
    data_theta    = sess['data_theta']

    # Boolean masks that select the target and non-target subsets of epochs
    t_mask  = labels == 1   # True for every target epoch row
    nt_mask = labels == 0   # True for every non-target epoch row

    # Resolve the 32-channel array index for Pz and AFz from their subset positions
    pz_idx  = delta_indices[pz_in_delta]
    afz_idx = theta_indices[afz_in_theta]

    fig, axes = plt.subplots(3, 1, figsize=(12, 10))
    fig.suptitle(
        f'Feature extraction — Subject {SUBJECT}, Session {SESSION} ({flash_ms} ms flash)',
        fontsize=13, fontweight='bold',
    )

    # ── Panel 1: ERP at Cz ───────────────────────────────────────────────────
    # data[t_mask, cz_idx, :] selects only target epochs, then the Cz channel.
    # .mean(0) averages across epochs (axis 0), leaving shape (n_times,).
    # * 1e6 converts Volts → µV for a human-readable scale.
    ax = axes[0]
    ax.plot(times, data[t_mask,  cz_idx, :].mean(0) * 1e6,
            color='red',  label=f'Target (n={t_mask.sum()})')
    ax.plot(times, data[nt_mask, cz_idx, :].mean(0) * 1e6,
            color='blue', label=f'Non-target (n={nt_mask.sum()})')
    ax.axvline(0, color='black', linestyle='--', alpha=0.6, linewidth=0.9)   # stimulus onset
    ax.axhline(0, color='gray',  linestyle=':',  linewidth=0.8)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude (µV)')
    ax.set_title('ERP at Cz — Target vs Non-target')
    ax.legend(fontsize=9)

    # ── Panel 2: Delta bandpass waveform at Pz ────────────────────────────────
    # Same averaging on the delta-filtered array — reveals the slow posterior wave
    ax = axes[1]
    ax.plot(times, data_delta[t_mask,  pz_idx, :].mean(0) * 1e6,
            color='red',  label=f'Target (n={t_mask.sum()})')
    ax.plot(times, data_delta[nt_mask, pz_idx, :].mean(0) * 1e6,
            color='blue', label=f'Non-target (n={nt_mask.sum()})')
    ax.axvline(0, color='black', linestyle='--', alpha=0.6, linewidth=0.9)
    ax.axhline(0, color='gray',  linestyle=':',  linewidth=0.8)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude (µV)')
    ax.set_title(f'Delta wave at Pz (0.5–3 Hz bandpass, order={BP_ORDER})')
    ax.legend(fontsize=9)

    # ── Panel 3: Theta bandpass waveform at AFz ───────────────────────────────
    # Frontal theta burst — oscillatory shape, peaks earlier than the slow delta wave
    ax = axes[2]
    ax.plot(times, data_theta[t_mask,  afz_idx, :].mean(0) * 1e6,
            color='red',  label=f'Target (n={t_mask.sum()})')
    ax.plot(times, data_theta[nt_mask, afz_idx, :].mean(0) * 1e6,
            color='blue', label=f'Non-target (n={nt_mask.sum()})')
    ax.axvline(0, color='black', linestyle='--', alpha=0.6, linewidth=0.9)
    ax.axhline(0, color='gray',  linestyle=':',  linewidth=0.8)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude (µV)')
    ax.set_title(f'Theta wave at AFz (3–7 Hz bandpass, order={BP_ORDER})')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()
    plt.close('all')   # free memory — avoids accumulating many figure objects when looping

## 6. Cross-Session Comparison

A 3×3 summary figure placing all three sessions side by side to reveal how **flash duration** affects the neural responses captured in each feature set.

**Why flash duration matters:**  
In Brain Invaders, each face is displayed for a fixed duration — 110 ms (Session 1, slow), 80 ms (Session 2, medium), or 50 ms (Session 3, fast). Shorter flashes leave less time for the visual stimulus to evoke a full P300 response. For the BCI to work reliably across all speeds, the ERP and band-limited features must remain discriminative even at 50 ms. This plot tests whether they do.

**What to look for across columns (left = slow, right = fast):**
- **Row 1 — ERP at Cz:** Does the P300 positive peak shrink, flatten, or shift in latency as the flash gets faster? A smaller peak at 50 ms suggests the sensory processing time is insufficient to generate a full P300.
- **Row 2 — Delta wave at Pz:** The slow posterior wave (500–900 ms) reflects higher-level cognitive processing and may be more robust to flash duration than the early sensory response.
- **Row 3 — Theta wave at AFz:** Frontal theta reflects working-memory updating — a process that starts *after* the stimulus is recognised. If the stimulus is too brief to be encoded, the theta burst may disappear at 50 ms.

**Why compare target vs non-target in each panel?**  
Each panel's red–blue gap is a direct visual proxy for how separable the two classes are in that feature. If the gap collapses at 50 ms (right column), classifiers trained on that feature will struggle at that session speed.

| Row | Signal | Electrode | Look for across sessions |
|-----|--------|-----------|--------------------------|
| 1 | ERP (broadband) | Cz | P300 amplitude vs flash speed |
| 2 | Delta wave (0.5–3 Hz) | Pz | Late slow wave stability |
| 3 | Theta wave (3–7 Hz) | AFz | Frontal burst vs flash speed |

In [ ]:
if len(all_sessions) < 3:
    print(f"Only {len(all_sessions)} session(s) loaded. All 3 are required for cross-session comparison.")
else:
    session_list = sorted(all_sessions.keys())   # [1, 2, 3] — sorted so columns go slow → fast

    pz_in_delta  = DELTA_CHANNELS.index('Pz')
    afz_in_theta = THETA_CHANNELS.index('AFz')

    # 3 rows (ERP / delta / theta) × 3 columns (one per session / flash duration)
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    fig.suptitle(
        f'Bandpass waveform comparison across flash durations — Subject {SUBJECT}',
        fontsize=14, fontweight='bold',
    )

    row_labels = ['ERP at Cz (µV)', 'Delta wave at Pz (µV)', 'Theta wave at AFz (µV)']
    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=10, labelpad=8)   # shared y-axis label on leftmost column only

    for col, SESSION in enumerate(session_list):   # col 0 = 110 ms, col 1 = 80 ms, col 2 = 50 ms
        sess          = all_sessions[SESSION]
        flash_ms      = sess['flash_ms']
        data          = sess['data']
        times         = sess['times']
        labels        = sess['labels']
        cz_idx        = sess['cz_idx']
        delta_indices = sess['delta_indices']
        theta_indices = sess['theta_indices']
        data_delta    = sess['data_delta']
        data_theta    = sess['data_theta']

        pz_idx  = delta_indices[pz_in_delta]
        afz_idx = theta_indices[afz_in_theta]

        t_mask  = labels == 1   # select target epochs
        nt_mask = labels == 0   # select non-target epochs

        # Row 0: ERP at Cz — broadband average (sanity check / baseline comparison)
        ax = axes[0, col]
        ax.plot(times, data[t_mask,  cz_idx, :].mean(0) * 1e6,
                color='red',  label='Target',     linewidth=1.2)
        ax.plot(times, data[nt_mask, cz_idx, :].mean(0) * 1e6,
                color='blue', label='Non-target', linewidth=1.2)
        ax.axvline(0, color='black', linestyle='--', alpha=0.5, linewidth=0.8)
        ax.axhline(0, color='gray',  linestyle=':',  linewidth=0.8)
        ax.set_title(f'{flash_ms} ms flash', fontweight='bold')   # column header = flash duration
        ax.set_xlabel('Time (s)')
        if col == 0:
            ax.legend(fontsize=7)   # legend only on leftmost column to avoid clutter

        # Row 1: Delta wave at Pz — slow posterior component
        ax = axes[1, col]
        ax.plot(times, data_delta[t_mask,  pz_idx, :].mean(0) * 1e6,
                color='red',  label='Target',     linewidth=1.2)
        ax.plot(times, data_delta[nt_mask, pz_idx, :].mean(0) * 1e6,
                color='blue', label='Non-target', linewidth=1.2)
        ax.axvline(0, color='black', linestyle='--', alpha=0.5, linewidth=0.8)
        ax.axhline(0, color='gray',  linestyle=':',  linewidth=0.8)
        ax.set_xlabel('Time (s)')
        if col == 0:
            ax.legend(fontsize=7)

        # Row 2: Theta wave at AFz — frontal attentional theta burst
        ax = axes[2, col]
        ax.plot(times, data_theta[t_mask,  afz_idx, :].mean(0) * 1e6,
                color='red',  label='Target',     linewidth=1.2)
        ax.plot(times, data_theta[nt_mask, afz_idx, :].mean(0) * 1e6,
                color='blue', label='Non-target', linewidth=1.2)
        ax.axvline(0, color='black', linestyle='--', alpha=0.5, linewidth=0.8)
        ax.axhline(0, color='gray',  linestyle=':',  linewidth=0.8)
        ax.set_xlabel('Time (s)')
        if col == 0:
            ax.legend(fontsize=7)

    plt.tight_layout()
    plt.show()
    plt.close('all')

## Summary

This notebook produced one `.npz` file per session containing:

| Key | Content |
|-----|---------|
| `erp` | Flattened post-stimulus waveform, all 32 ch, ds×16 |
| `delta_wave` | Bandpass 0.5–3 Hz (butter order 4, filtfilt), 9 ch (Cz, CP1/2/5/6, Pz, P3/4/8), ds×16 |
| `theta_wave` | Bandpass 3–7 Hz (butter order 4, filtfilt), 6 ch (AFz, FC1/2, F3/4, Cz), ds×16 |
| `dt_wave` | delta_wave + theta_wave concatenated |
| `labels` | Binary target labels aligned to epochs |
| `flash_ms` | Flash duration for that session |
| `subject` | Subject number |
| `session` | Session number |

Load in the next notebook:
```python
d = np.load('subject_02_session_01_features.npz')
X_erp,        y = d['erp'],        d['labels']
X_delta_wave, y = d['delta_wave'], d['labels']
X_theta_wave, y = d['theta_wave'], d['labels']
X_dt_wave,    y = d['dt_wave'],    d['labels']
```

**Next step:** `03_classification.ipynb` — train and evaluate classifiers on each feature set, cross-validated across subjects and flash durations.